# Chapter 32 – Lab: compile time, batch size and the recompilation trigger

One section per step of the [Chapter 32 lab](../index.md#lab). Each step has a Markdown cell with guidance and
an empty code cell for your work. This notebook needs ToOp: run it in the ToOp container (or with ToOp's
virtual environment) from this folder. The chapter's module `solver_lab_toop.py` is next to it: import it
instead of copying its code. The written solutions are in [solutions.md](../solutions.md#lab).

In [ ]:
import jax
import solver_lab_toop as lab

print('JAX', jax.__version__, 'on', jax.default_backend(), '- x64 enabled:', jax.config.jax_enable_x64)
print(lab.__doc__)

## Step 1 – Setup

`lab.setup(folder)` writes `case57_data_powsybl` into `folder` when it is missing, calls `load_grid` and
`run_initial_loadflow`, and returns ToOp's `StaticInformation`. Keep `sc = si.solver_config` and
`di = si.dynamic_information`, and build `topologies = lab.default_topology(sc, batch_size=4)`. Always wait for
results with `jax.block_until_ready` before stopping a timer (`lab.timed` does that for you).

In [ ]:
si = lab.setup(lab.grid_folder('case57'))
sc, di = si.solver_config, si.dynamic_information
topologies = lab.default_topology(sc, batch_size=4)
print(di.n_nodes, 'nodes,', di.n_branches, 'branches,', sc.n_sub_relevant, 'relevant substations')
# your work here

## Step 2 – Compile vs run

Time `lab.run_solver(topologies, None, None, di, sc)` three times. Then call `lab.run_solver_symmetric` with a
pass-through aggregation function defined once (`lab.passthrough`) and with a fresh `lambda` on every call.
Tabulate the first-call, cached and fresh-lambda times and explain the pattern. `lab.compile_vs_run(si)` is the
reference implementation; `lab.compiled_programs()` counts the compiled programs of the solver core.

In [ ]:
# your work here

## Step 3 – Find the trigger

For each change below, first predict "recompiles", "reuses correctly" or "reuses incorrectly", then measure
the time and inspect `result.n_1_results.pf_n_1_max` (shape and maximum):

- `replace(sc)` with no field changed;
- `replace(sc, number_most_affected=20)`, and separately the in-place `sc.number_most_affected = 20`;
- a new `DynamicInformation` with `nodal_injections` scaled by 1.1 (the flows do not change: find in
  `dc_solver/jax/compute_batch.py` which field the N-0 flows are updated from, and scale that one too);
- a new `DynamicInformation` whose `branch_limits.max_mw_flow` is halved, passed with the *same* `sc`,
  and then with `replace(sc)`;
- a batch of 8 topologies instead of 4. The naive prediction is "recompiles" (a new array shape); it
  does not, because `run_solver_symmetric` pads every batch to a multiple of `batch_size_bsdf` (16 by
  default), so 4 and 8 topologies reach the jitted core with identical shapes. Confirm it, and say
  what batch size would make it recompile.

To tell a cached call from a recompile, count the compiled programs of the jitted core before and after
each call, `iterate_symmetric_sequential._cache_size()` (`dc_solver/jax/topology_looper.py`): a cache
hit leaves it unchanged. Do not wrap `run_solver` in `with jax.no_tracing():` under ToOp's JAX 0.5.3:
the eager Python glue around the core (padding, result storage, the final slicing) dispatches
primitives that this version reports as "re-tracing", so the context manager raises on a cache hit as
well; it is only usable around the jitted core itself, as
`packages/topology_optimizer_pkg/tests/test_jax_compilation.py` does. Explain every result from
`static_argnames`, `SolverConfig.__hash__` and the attributes of `DefaultAggregateOutputFn`. On ToOp
commit `ba1874e` (CPU, JAX 0.5.3), halving the limits with the same
`sc` returned the *old* relative loadings with no recompile, while `replace(sc)` recompiled and doubled
them: write this up as a bug report with a minimal reproduction.

In this notebook, make the in-place change on a copy, `sc2 = replace(sc)`, so the rest of the notebook keeps a
clean `sc`; build the new `DynamicInformation` with `eqx.tree_at` or `dataclasses.replace`.
`lab.compiled_programs()` returns `iterate_symmetric_sequential._cache_size()`, and `lab.trigger_study(si)` is
the reference.

In [ ]:
# your work here

## Step 4 – Batch size and memory

On `case300_powsybl` (`lab.setup(lab.grid_folder('case300'), 'case300')`), sweep `batch_size_bsdf` and
`batch_size_injection` over powers of two with a fixed number of topologies (`replace(sc, batch_size_bsdf=...)`).
Record compile time, topologies per second and peak memory: `nvidia-smi` on a GPU with
`XLA_PYTHON_CLIENT_PREALLOCATE=false`, `docker stats` on CPU. `lab.batch_sweep(si300)` records the process's
RSS growth as a CPU stand-in. Draw a heat map as in `dc_solver/jax/parse_batch_size_finder.ipynb`, find the
out-of-memory boundary on the GPU, and repeat one point in float32 for the float64 penalty of your device.
ToOp switches `jax_enable_x64` on when it is imported, so `JAX_ENABLE_X64=0` alone changes nothing:
`gpu_compare_toop.to_float32` narrows the inputs before x64 is switched off, and
`python gpu_compare_toop.py sweep ROOT float32` runs every point in a fresh process (`JAX_PLATFORMS=cuda,cpu`
for the GPU, `cpu` for the CPU).

In [ ]:
# your work here